# Doc-to-LoRA: запуск и анализ

**Репозиторий:** https://github.com/SakanaAI/doc-to-lora  
**Статья:** https://arxiv.org/abs/2602.15902  

## Подготовка
1. Runtime → Change runtime type → **T4 GPU**
2. Создай токен: https://huggingface.co/settings/tokens (тип **Read**)
3. Colab → Secrets → добавь `HF_TOKEN`

## Секция 0: Проверка GPU

In [1]:
!nvidia-smi
import torch
print('torch:', torch.__version__)
print('CUDA:', torch.version.cuda)
print('GPU:', torch.cuda.get_device_name(0))
print('Compute:', torch.cuda.get_device_capability())
print('VRAM:', round(torch.cuda.get_device_properties(0).total_memory / 1e9, 1), 'GB')

Thu Mar 12 19:30:04 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   63C    P8             10W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

## Секция 1: Клонирование репозитория

In [2]:
import os
os.environ['PATH'] = '/root/.local/bin:' + os.environ['PATH']

if not os.path.exists('/content/doc-to-lora'):
    %cd /content
    !git clone https://github.com/SakanaAI/doc-to-lora.git

%cd /content/doc-to-lora
!pwd && ls

/content
Cloning into 'doc-to-lora'...
remote: Enumerating objects: 118, done.
remote: Counting objects: 100% (49/49), done.
remote: Compressing objects: 100% (47/47), done.
remote: Total 118 (delta 11), reused 2 (delta 2), pack-reused 69 (from 2)
Receiving objects: 100% (118/118), 7.04 MiB | 26.91 MiB/s, done.
Resolving deltas: 100% (13/13), done.
/content/doc-to-lora
/content/doc-to-lora
accelerate_config.yaml	data	    LICENSE	    scripts   uv.lock
assets			demo	    pyproject.toml  setup.py  watcher.py
chat_templates		examples    README.md	    src       webui
configs			install.sh  run_eval.py     train.py


## Секция 2: Установка зависимостей

Занимает ~3-5 минут. Запускать один раз за сессию.

In [3]:
import os
os.environ['PATH'] = '/root/.local/bin:' + os.environ['PATH']
%cd /content/doc-to-lora

!curl -LsSf https://astral.sh/uv/install.sh | sh
!uv self update
!uv venv --python 3.10 --seed
!uv pip install torch==2.6.0 torchvision==0.21.0 torchaudio==2.6.0 --torch-backend=cu124
!uv sync
!uv pip install flashinfer-python==0.2.2 -i https://flashinfer.ai/whl/cu124/torch2.6
!.venv/bin/pip install tokenizers==0.21.0
print('\n✅ Installation complete')

/content/doc-to-lora
downloading uv 0.10.9 x86_64-unknown-linux-gnu
no checksums to verify
installing to /usr/local/bin
  uv
  uvx
everything's installed!
info: Checking for updates...
success: You're on the latest version of uv (v0.10.9)
Using CPython 3.10.12 interpreter at: /usr/bin/python3.10
Creating virtual environment with seed packages at: .venv
 + packaging==26.0
 + pip==26.0.1
 + setuptools==82.0.1
 + wheel==0.46.3
Activate with: source .venv/bin/activate
Using Python 3.12.12 environment at: /usr
Resolved 28 packages in 818ms
Prepared 18 packages in 38.63s
Uninstalled 18 packages in 963ms
Installed 18 packages in 271ms
 - nvidia-cublas-cu12==12.8.4.1
 + nvidia-cublas-cu12==12.4.5.8
 - nvidia-cuda-cupti-cu12==12.8.90
 + nvidia-cuda-cupti-cu12==12.4.127
 - nvidia-cuda-nvrtc-cu12==12.8.93
 + nvidia-cuda-nvrtc-cu12==12.4.127
 - nvidia-cuda-runtime-cu12==12.8.90
 + nvidia-cuda-runtime-cu12==12.4.127
 - nvidia-cudnn-cu12==9.10.2.21
 + nvidia-cudnn-cu12==9.1.0.70
 - nvidia-cufft-cu12

## Секция 3: Патчи для T4 GPU

T4 имеет compute capability 7.5, FlashAttention-2 требует 8.0+.  
Применяем 5 патчей для переключения на eager attention:

| # | Файл | Проблема | Патч |
|---|------|---------|------|
| 1 | `hypernet.py` | `use_flash_attn=True` при вызове get_model | → `False` |
| 2 | `idefics2.py` | `eager` закомментирован в ATTENTION_CLASSES | раскомментировать |
| 3 | `idefics2.py` | `Idefics2Perceiver.__init__` не форсирует eager | → установить в конфиге |
| 4 | `idefics2.py` | `assert self._use_flash_attention_2` падает | → `False` |
| 5 | `idefics2.py` | `unpad_input` из flash_attn не доступна | → ручной расчёт |

In [24]:
import sys, os, glob
REPO = '/content/doc-to-lora'
sys.path.insert(0, f'{REPO}/src')

venv_libs = glob.glob(f'{REPO}/.venv/lib/python*/site-packages')
if venv_libs:
    with open(f'{venv_libs[0]}/ctx_to_lora.pth', 'w') as f:
        f.write(f'{REPO}/src\n')
    print(f'.pth written to {venv_libs[0]}')

# ── P1: hypernet.py ──
hp = f'{REPO}/src/ctx_to_lora/modeling/hypernet.py'
with open(hp) as f: c = f.read()
if 'use_flash_attn=False,  # T4' not in c:
    c = c.replace(
        '            use_flash_attn=use_flash_attn,\n        )',
        '            use_flash_attn=False,  # T4 patch\n        )'
    )
    with open(hp, 'w') as f: f.write(c)
    print('P1 OK')
else: print('P1 skip')

# ── P2-P6: idefics2.py ──
ip = f'{REPO}/src/ctx_to_lora/modeling/idefics2.py'
with open(ip) as f: c = f.read()
n = 0

# P2
if '# "eager": Idefics2PerceiverAttention,' in c:
    c = c.replace('    # "eager": Idefics2PerceiverAttention,',
                  '    "eager": Idefics2PerceiverAttention,')
    n += 1; print('P2 OK')
else: print('P2 skip')

# P3
O3 = """    def __init__(
        self,
        encoder_config: Idefics2PerceiverConfig,
        decoder_config: Idefics2PerceiverConfig,
    ):
        super().__init__(encoder_config)"""
N3 = """    def __init__(
        self,
        encoder_config: Idefics2PerceiverConfig,
        decoder_config: Idefics2PerceiverConfig,
    ):
        encoder_config._attn_implementation = "eager"  # T4
        decoder_config._attn_implementation = "eager"  # T4
        super().__init__(encoder_config)"""
if O3 in c:
    c = c.replace(O3, N3); n += 1; print('P3 OK')
else: print('P3 skip')

# P4
if 'assert self._use_flash_attention_2' in c:
    c = c.replace(
        '        self._use_flash_attention_2 = config._attn_implementation == "flash_attention_2"\n        assert self._use_flash_attention_2',
        '        self._use_flash_attention_2 = False  # T4 patch'
    )
    n += 1; print('P4 OK')
else: print('P4 skip')

# P5
O5 = """        latents, self_attn_weights, present_key_value = self.self_attn(
            latents=latents,
            is_cross_attn=self.is_cross_attn,
            context=context,
            attention_mask=attention_mask,
            position_ids=position_ids,
            **kwargs,
        )"""
N5 = """        # T4: eager attention
        if self.is_cross_attn:
            latents, self_attn_weights, present_key_value = self.self_attn(
                latents=latents, context=context, attention_mask=attention_mask,
                position_ids=position_ids, past_key_value=past_key_value,
                output_attentions=output_attentions, use_cache=use_cache,
            )
        else:
            latents, self_attn_weights, present_key_value = self.self_attn(
                latents=latents, context=latents, attention_mask=None,
                position_ids=position_ids, past_key_value=past_key_value,
                output_attentions=output_attentions, use_cache=use_cache,
            )"""
if 'is_cross_attn=self.is_cross_attn,' in c:
    c = c.replace(O5, N5); n += 1; print('P5 OK')
else: print('P5 skip')

# P6: replace the ENTIRE block from mask prep through layer loop
O6_START = '        attention_mask = (\n            _prepare_4d_attention_mask('
O6_END = '            layer_outputs = layer(**attn_kwargs)'
if O6_START in c and O6_END in c:
    i_start = c.index(O6_START)
    i_end = c.index(O6_END) + len(O6_END)
    N6 = """        # ── T4: eager forward ──
        if attention_mask is not None:
            latent_ones = torch.ones(
                attention_mask.size(0), self.n_latents,
                device=attention_mask.device, dtype=attention_mask.dtype
            )
            full_mask_2d = torch.cat([attention_mask, latent_ones], dim=1)
            attention_mask = _prepare_4d_attention_mask(
                full_mask_2d, latents.dtype, tgt_len=self.n_latents
            )

        compressed_context = latents

        for i, layer in enumerate(self.layers):
            if layer.is_cross_attn:
                layer_outputs = layer(
                    latents=compressed_context, context=context,
                    attention_mask=attention_mask,
                    past_key_value=None, output_attentions=False, use_cache=False,
                )
            else:
                layer_outputs = layer(
                    latents=compressed_context, context=compressed_context,
                    past_key_value=None, output_attentions=False, use_cache=False,
                )"""
    c = c[:i_start] + N6 + c[i_end:]
    n += 1; print('P6 OK')
else: print('P6 skip')

with open(ip, 'w') as f: f.write(c)
print(f'\n✅ Done ({n} applied)')


.pth written to /content/doc-to-lora/.venv/lib/python3.10/site-packages
P1 OK
P2 OK
P3 OK
P4 OK
P5 OK
P6 OK

✅ Done (5 applied)


## Секция 4: Проверка импортов

In [5]:
import os
os.environ['PATH'] = '/root/.local/bin:' + os.environ['PATH']

with open('/tmp/test_import.py', 'w') as f:
    f.write('import sys\n')
    f.write('sys.path.insert(0, "/content/doc-to-lora/src")\n')
    f.write('from ctx_to_lora.model_loading import get_tokenizer\n')
    f.write('from ctx_to_lora.modeling.hypernet import ModulatedPretrainedModel\n')
    f.write('print("imports OK")\n')

%cd /content/doc-to-lora
!uv run python /tmp/test_import.py

/content/doc-to-lora
[2026-03-12 19:38:47,023] [INFO] [real_accelerator.py:254:get_accelerator] Setting ds_accelerator to cuda (auto detect)
[2026-03-12 19:38:50,906] [INFO] [logging.py:107:log_dist] [Rank -1] [TorchCheckpointEngine] Initialized with serialization = False
imports OK


## Секция 5: Авторизация HuggingFace

In [6]:
import os
try:
    from google.colab import userdata
    hf_token = userdata.get('HF_TOKEN')
    print('Token loaded from Colab Secrets')
except:
    import getpass
    hf_token = getpass.getpass('HuggingFace token: ')
os.environ['HF_TOKEN'] = hf_token

%cd /content/doc-to-lora
!uv run huggingface-cli login --token $HF_TOKEN

Token loaded from Colab Secrets
/content/doc-to-lora
⚠️  Warning: 'huggingface-cli login' is deprecated. Use 'hf auth login' instead.
The token has not been saved to the git credentials helper. Pass `add_to_git_credential=True` in this function directly or `--add-to-git-credential` if using via `hf`CLI if you want to set the git credential as well.
Token is valid (permission: read).
The token `t2l` has been saved to /root/.cache/huggingface/stored_tokens
Your token has been saved to /root/.cache/huggingface/token
Login successful.
Note: Environment variable`HF_TOKEN` is set and is the current active token independently from the token you've just configured.


## Секция 6: Скачивание чекпоинтов

Доступные модели:
- `gemma_2b_d2l` — Gemma-2-2b-it, **1.26 GB** ✅ рекомендуется для T4
- `gemma_demo` — дополнительно обученная Gemma, **1.26 GB**
- `qwen_4b_d2l` — Qwen-2.5-3B, 1.68 GB
- `mistral_7b_d2l` — Mistral-7B, **2.30 GB** ❌ не влезет в T4

In [7]:
import os
os.environ['PATH'] = '/root/.local/bin:' + os.environ['PATH']
%cd /content/doc-to-lora

!uv run huggingface-cli download SakanaAI/doc-to-lora \
    --local-dir trained_d2l \
    --include "gemma_2b_d2l/*" "gemma_demo/*"

!ls trained_d2l/
!ls trained_d2l/gemma_2b_d2l/

/content/doc-to-lora
⚠️  Warning: 'huggingface-cli download' is deprecated. Use 'hf download' instead.
Fetching 4 files:   0% 0/4 [00:00<?, ?it/s]Downloading 'gemma_demo/checkpoint-80000/pytorch_model.bin' to 'trained_d2l/.cache/huggingface/download/gemma_demo/checkpoint-80000/Q1p2l2BzM1m6P5jKvr8WTq1TUio=.b0e94a7200dfb7d1dd65b0d7d1fc95088f5bf2560c3ec07114e2e92069730d82.incomplete'

gemma_demo/checkpoint-80000/pytorch_mode(…):   0% 0.00/1.26G [00:00<?, ?B/s]

gemma_2b_d2l/checkpoint-20000/pytorch_mo(…):   0% 0.00/1.26G [00:00<?, ?B/s]


args.yaml: 5.41kB [00:00, 8.01MB/s]
Download complete. Moving file to trained_d2l/gemma_demo/args.yaml



args.yaml: 5.64kB [00:00, 15.3MB/s]
Download complete. Moving file to trained_d2l/gemma_2b_d2l/args.yaml
Fetching 4 files:  25% 1/4 [00:00<00:02,  1.30it/s]

gemma_2b_d2l/checkpoint-20000/pytorch_mo(…):   0% 600k/1.26G [00:01<56:51, 369kB/s]
gemma_demo/checkpoint-80000/pytorch_mode(…):   0% 739k/1.26G [00:01<50:31, 416kB/s]
gemma_demo/checkpoint-8000

## Секция 7: Загрузка модели

Загружаем `gemma_2b_d2l` — обучена на QA датасетах (SQuAD, DROP, ROPES, PwC).  
Архитектура D2L: context encoder (активации Gemma-2) → Perceiver Resampler → LoRA на down_proj.

In [9]:
import os
os.environ['PATH'] = '/root/.local/bin:' + os.environ['PATH']

with open('/tmp/load_model.py', 'w') as f:
    f.write('import sys\nsys.path.insert(0, "/content/doc-to-lora/src")\nimport torch\nfrom ctx_to_lora.model_loading import get_tokenizer\nfrom ctx_to_lora.modeling.hypernet import ModulatedPretrainedModel\n\nCKPT = "trained_d2l/gemma_2b_d2l/checkpoint-20000/pytorch_model.bin"\nprint("Loading state dict...")\nstate_dict = torch.load(CKPT, weights_only=False, map_location="cuda")\nprint(f"State dict keys: {len(state_dict)}")\n\nprint("Building model...")\nmodel = ModulatedPretrainedModel.from_state_dict(\n    state_dict, train=False, use_sequence_packing=False\n)\nmodel.reset()\ntokenizer = get_tokenizer(model.base_model.name_or_path)\n\nprint(f"Base model: {model.base_model.name_or_path}")\nprint(f"Device: {next(model.parameters()).device}")\nprint(f"VRAM used: {torch.cuda.memory_allocated() / 1e9:.2f} GB")\nprint("Model ready!")')

%cd /content/doc-to-lora
!uv run python /tmp/load_model.py

/content/doc-to-lora
[2026-03-12 19:41:25,739] [INFO] [real_accelerator.py:254:get_accelerator] Setting ds_accelerator to cuda (auto detect)
[2026-03-12 19:41:27,439] [INFO] [logging.py:107:log_dist] [Rank -1] [TorchCheckpointEngine] Initialized with serialization = False
Loading state dict...
State dict keys: 143
Building model...
lora_config: LoraConfig(task_type='CAUSAL_LM', peft_type=<PeftType.LORA: 'LORA'>, auto_mapping=None, base_model_name_or_path='google/gemma-2-2b-it', revision=None, inference_mode=False, r=8, target_modules={'down_proj'}, exclude_modules=None, lora_alpha=45.254833995939045, lora_dropout=0.0, fan_in_fan_out=False, bias='none', use_rslora=False, modules_to_save=None, init_lora_weights=True, layers_to_transform=None, layers_pattern=None, rank_pattern={}, alpha_pattern={}, megatron_config=None, megatron_core='megatron.core', trainable_token_indices=None, loftq_config={}, eva_config=None, corda_config=None, use_dora=False, layer_replication=None, runtime_config=Lo

## Секция 8: Демонстрация internalize

Ключевая особенность D2L: `model.internalize(doc)` — один forward pass гиперсети,  
после которого модель отвечает на вопросы о документе без RAG и без расширения контекста.

In [25]:
import os
os.environ['PATH'] = '/root/.local/bin:' + os.environ['PATH']

with open('/tmp/run_d2l.py', 'w') as f:
    f.write('import sys\nsys.path.insert(0, "/content/doc-to-lora/src")\nimport torch\nfrom ctx_to_lora.model_loading import get_tokenizer\nfrom ctx_to_lora.modeling.hypernet import ModulatedPretrainedModel\n\nCKPT = "trained_d2l/gemma_2b_d2l/checkpoint-20000/pytorch_model.bin"\nstate_dict = torch.load(CKPT, weights_only=False, map_location="cuda")\nmodel = ModulatedPretrainedModel.from_state_dict(state_dict, train=False, use_sequence_packing=False)\nmodel.reset()\ntokenizer = get_tokenizer(model.base_model.name_or_path)\n\ndef ask(question, doc=None, max_new_tokens=150):\n    if doc: model.internalize(doc)\n    else: model.reset()\n    chat = [{"role": "user", "content": question}]\n    ids = tokenizer.apply_chat_template(\n        chat, add_special_tokens=False, return_attention_mask=False,\n        add_generation_prompt=True, return_tensors="pt"\n    ).to(model.device)\n    with torch.no_grad():\n        out = model.generate(input_ids=ids, max_new_tokens=max_new_tokens)\n    return tokenizer.decode(out[0][ids.shape[1]:], skip_special_tokens=True)\n\ndoc = open("data/sakana_wiki.txt").read()\nprint(f"Document ({len(doc)} chars):\\n{doc[:300]}...\\n")\n\nq = "What is Sakana AI and what are its main research directions?"\n\nprint("=" * 60)\nprint("WITHOUT internalization:")\nprint("=" * 60)\nprint(ask(q))\n\nprint()\nprint("=" * 60)\nprint("WITH internalization:")\nprint("=" * 60)\nprint(ask(q, doc=doc))')

%cd /content/doc-to-lora
!uv run python /tmp/run_d2l.py

/content/doc-to-lora
[2026-03-12 20:06:00,475] [INFO] [real_accelerator.py:254:get_accelerator] Setting ds_accelerator to cuda (auto detect)
[2026-03-12 20:06:02,866] [INFO] [logging.py:107:log_dist] [Rank -1] [TorchCheckpointEngine] Initialized with serialization = False
lora_config: LoraConfig(task_type='CAUSAL_LM', peft_type=<PeftType.LORA: 'LORA'>, auto_mapping=None, base_model_name_or_path='google/gemma-2-2b-it', revision=None, inference_mode=False, r=8, target_modules={'down_proj'}, exclude_modules=None, lora_alpha=45.254833995939045, lora_dropout=0.0, fan_in_fan_out=False, bias='none', use_rslora=False, modules_to_save=None, init_lora_weights=True, layers_to_transform=None, layers_pattern=None, rank_pattern={}, alpha_pattern={}, megatron_config=None, megatron_core='megatron.core', trainable_token_indices=None, loftq_config={}, eva_config=None, corda_config=None, use_dora=False, layer_replication=None, runtime_config=LoraRuntimeConfig(ephemeral_gpu_offload=False), lora_bias=False

## Секция 9: Тест с произвольными документами

In [26]:
import os
os.environ['PATH'] = '/root/.local/bin:' + os.environ['PATH']

with open('/tmp/custom_docs.py', 'w') as f:
    f.write('import sys\nsys.path.insert(0, "/content/doc-to-lora/src")\nimport torch\nfrom ctx_to_lora.model_loading import get_tokenizer\nfrom ctx_to_lora.modeling.hypernet import ModulatedPretrainedModel\n\nCKPT = "trained_d2l/gemma_2b_d2l/checkpoint-20000/pytorch_model.bin"\nstate_dict = torch.load(CKPT, weights_only=False, map_location="cuda")\nmodel = ModulatedPretrainedModel.from_state_dict(state_dict, train=False, use_sequence_packing=False)\nmodel.reset()\ntokenizer = get_tokenizer(model.base_model.name_or_path)\n\ndef ask(question, doc=None, max_new_tokens=150):\n    if doc: model.internalize(doc)\n    else: model.reset()\n    chat = [{"role": "user", "content": question}]\n    ids = tokenizer.apply_chat_template(\n        chat, add_special_tokens=False, return_attention_mask=False,\n        add_generation_prompt=True, return_tensors="pt"\n    ).to(model.device)\n    with torch.no_grad():\n        out = model.generate(input_ids=ids, max_new_tokens=max_new_tokens)\n    return tokenizer.decode(out[0][ids.shape[1]:], skip_special_tokens=True)\n\ndoc1 = ("LoRA (Low-Rank Adaptation) is a parameter-efficient fine-tuning method. "\n        "Instead of updating all weights, LoRA adds small matrices A and B to each layer. "\n        "The weight update is delta_W = B @ A * (alpha / r). "\n        "Typical rank values: r=4, r=8, r=16. "\n        "LoRA was introduced by Hu et al. in 2021.")\n\nq1 = "How does LoRA compute weight updates and who introduced it?"\nprint("Doc 1: LoRA technical description")\nprint(f"Q: {q1}")\nprint(f"Without: {ask(q1)[:200]}")\nprint(f"With:    {ask(q1, doc=doc1)[:200]}")\n\nprint()\n\ndoc2 = ("The Aurora research station was established in 2024 by the Nordic Consortium. "\n        "It houses 14 researchers with maximum capacity of 25 people. "\n        "The station specializes in permafrost monitoring and methane tracking. "\n        "Resupply happens every 6 weeks by helicopter.")\n\nq2 = "How many researchers does the Aurora station house and how often is it resupplied?"\nprint("Doc 2: fictional research station")\nprint(f"Q: {q2}")\nprint(f"Without: {ask(q2)[:200]}")\nprint(f"With:    {ask(q2, doc=doc2)[:200]}")')

%cd /content/doc-to-lora
!uv run python /tmp/custom_docs.py

/content/doc-to-lora
[2026-03-12 20:06:54,181] [INFO] [real_accelerator.py:254:get_accelerator] Setting ds_accelerator to cuda (auto detect)
[2026-03-12 20:06:56,408] [INFO] [logging.py:107:log_dist] [Rank -1] [TorchCheckpointEngine] Initialized with serialization = False
lora_config: LoraConfig(task_type='CAUSAL_LM', peft_type=<PeftType.LORA: 'LORA'>, auto_mapping=None, base_model_name_or_path='google/gemma-2-2b-it', revision=None, inference_mode=False, r=8, target_modules={'down_proj'}, exclude_modules=None, lora_alpha=45.254833995939045, lora_dropout=0.0, fan_in_fan_out=False, bias='none', use_rslora=False, modules_to_save=None, init_lora_weights=True, layers_to_transform=None, layers_pattern=None, rank_pattern={}, alpha_pattern={}, megatron_config=None, megatron_core='megatron.core', trainable_token_indices=None, loftq_config={}, eva_config=None, corda_config=None, use_dora=False, layer_replication=None, runtime_config=LoraRuntimeConfig(ephemeral_gpu_offload=False), lora_bias=False